In [1]:
import os
import subprocess
import logging
import sys

def source_lmod_script(script_path):
    """
    Source an Lmod/module script and import environment variables into Python safely,
    suppressing terminal warnings.
    """
    # Use a non-interactive login shell (bash -l), redirect errors
    command = f'bash -l -c "source {script_path} >/dev/null 2>&1; printenv -0"'
    
    proc = subprocess.Popen(command, stdout=subprocess.PIPE, shell=True)
    out, _ = proc.communicate()
    
    # Parse null-separated environment variables
    for env_var in out.split(b'\0'):
        if env_var:
            key, _, value = env_var.partition(b'=')
            os.environ[key.decode()] = value.decode()

# Example usage
M3_BUILD_DIR = "/home/henryi/scratch/venvs/.venv_sbi/bin/"
TUTORIAL_BUILD_DIR = M3_BUILD_DIR
source_lmod_script(f"{M3_BUILD_DIR}/setup.MaCh3.sh")
source_lmod_script(f"{TUTORIAL_BUILD_DIR}/setup.MaCh3Tutorial.sh")
os.environ["OMP_NUM_THREADS"] = "8"


my_stderr = sys.stderr = open('errors.txt', 'w')  # redirect stderr to file
get_ipython().log.handlers[0].stream = my_stderr  # log errors to new stderr
get_ipython().log.setLevel(logging.INFO)  # errors are logged at info level

cat: write error: Broken pipe
cat: write error: Broken pipe


In [2]:
from mach3sbitools.inference.sbi_interface import MaCh3SBIInterface
from mach3sbitools.mach3_interface.mach3_simulator import MaCh3Simulator
from mach3sbitools.utils.logger import MaCh3Logger, get_logger
from mach3sbitools.utils.config import TrainingConfig, PosteriorConfig

import pickle as pkl
from pathlib import Path
import uproot as ur
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib import pyplot as plt
from tqdm import tqdm

import torch
import numpy as np

logger = get_logger("mach3sbitools")
log_level='INFO'
log_file=None
MaCh3Logger("mach3sbitools", level=log_level, log_file=log_file)

In [3]:
inference = MaCh3SBIInterface("Tutorial", Path("/home/henryi/sft/MaCh3Tutorial/TutorialConfigs/FitterConfig.yaml"),
                              cyclical_pars=['delta_cp'])

[Monitor.cpp][info] ##################################
[Monitor.cpp][info] Welcome to:  
[Monitor.cpp][info]   __  __        _____ _     ____  
[Monitor.cpp][info]  |  \/  |      / ____| |   |___ \ 
[Monitor.cpp][info]  | \  / | __ _| |    | |__   __) |
[Monitor.cpp][info]  | |\/| |/ _` | |    | '_ \ |__ < 
[Monitor.cpp][info]  | |  | | (_| | |____| | | |___) |
[Monitor.cpp][info]  |_|  |_|\__,_|\_____|_| |_|____/ 
[Monitor.cpp][info] Version: 2.3.1
[Monitor.cpp][info] ##################################
[Monitor.cpp][info] Using following CPU:
[Monitor.cpp][info] model name	: AMD EPYC 9454 48-Core Processor
[Monitor.cpp][info] cpu MHz		: 3799.297
[Monitor.cpp][info] Architecture:                         x86_64
[Monitor.cpp][info] L1d cache:                            1.5 MiB (48 instances)
[Monitor.cpp][info] L1i cache:                            1.5 MiB (48 instances)
[Monitor.cpp][info] L2 cache:                             48 MiB (48 instances)
[Monitor.cpp][info] L3 cache:         

In [5]:
samples = {}
for i in range(100, 20600, 1000):
    # model_path=Path(f"/home/henryi/scratch/TutorialTestSBI/models/tutorial_model_resumed/2026-02-25-01-10_model_resumed/tutorial_model_epoch{i}.ts")
    # model_path = Path(f"/home/henryi/scratch/TutorialTestSBI/models/tutorial_with_ema/2026-02-25-06-03_ema_model/tutorial_model_epoch{i}.ts")
    model_path = Path(f"/home/henryi/scratch/TutorialTestSBI/best_models/256_hidden_20_transforms/tutorial_model_epoch{i}.ts")
    posterior_config = PosteriorConfig(
        hidden_features = 256,      # was 50 — scale up to match data volume
        num_transforms= 20,         # was 10 — fewer needed with NSF, less sequential overhead
        dropout_probability= 0.1,  # fine as-is, good regularisation for 5M samples  
        num_blocks= 2,             # fine as-is
        # NSF-specific
        num_bins= 10,             
        # spline bins, 8-12 is the usual sweet spot
    )
    
    inference.load_posterior(model_path, posterior_config)
    samples[i] = inference.sample_posterior(1000000).cpu()

2026-02-26 00:28:54 INFO     Loading autosave checkpoint from epoch 100

                    INFO     NPE created | NSF | hidden=256 transforms=20 blocks=2 bins=10

                    INFO     Density estimator loaded from                                                         
                             /home/henryi/scratch/TutorialTestSBI/best_models/256_hidden_20_transforms/tutorial_mod
                             el_epoch100.ts

                    INFO     Sampling 1,000,000 points from posterior

  0%|          | 0/1000000 [00:00<?, ?it/s]

...

2026-02-26 00:41:25 INFO     Loading autosave checkpoint from epoch 20100

                    INFO     NPE created | NSF | hidden=256 transforms=20 blocks=2 bins=10

                    INFO     Density estimator loaded from                                                         
                             /home/henryi/scratch/TutorialTestSBI/best_models/256_hidden_20_transforms/tutorial_mod
                             el_epoch20100.ts

                    INFO     Sampling 1,000,000 points from posterior

  0%|          | 0/1000000 [00:00<?, ?it/s]

In [6]:
model_path = Path(f"/home/henryi/scratch/TutorialTestSBI/best_models/256_hidden_20_transforms/tutorial_model_epoch20500.ts")
posterior_config = PosteriorConfig(
    hidden_features = 256,      # was 50 — scale up to match data volume
    num_transforms= 20,         # was 10 — fewer needed with NSF, less sequential overhead
    dropout_probability= 0.1,  # fine as-is, good regularisation for 5M samples  
    num_blocks= 2,             # fine as-is
    # NSF-specific
    num_bins= 10,             
    # spline bins, 8-12 is the usual sweet spot
)

inference.load_posterior(model_path, posterior_config)
samples[20500] = inference.sample_posterior(1000000).cpu()

2026-02-26 00:42:01 INFO     Loading autosave checkpoint from epoch 20500

                    INFO     NPE created | NSF | hidden=256 transforms=20 blocks=2 bins=10

                    INFO     Density estimator loaded from                                                         
                             /home/henryi/scratch/TutorialTestSBI/best_models/256_hidden_20_transforms/tutorial_mod
                             el_epoch20500.ts

                    INFO     Sampling 1,000,000 points from posterior

  0%|          | 0/1000000 [00:00<?, ?it/s]

In [7]:
input_tree = "/home/henryi/sft/MaCh3Tutorial/Test_NoAdapt.root"
mach3_chain = ur.open(f"{input_tree}:posteriors").arrays(library="np")

In [8]:

def cyclical_to_flat_weights(theta_cyc: torch.Tensor) -> np.ndarray:
    """
    Args:
        theta_cyc: shape (n_samples,) for a single cyclical parameter
    """
    w = 1.0 / (2.0 * torch.sin((theta_cyc + 2.0 * np.pi) / 4.0) ** 2)
    return w.cpu().numpy()

# 4. Normalise weights
# dcp_index = inference.simulator.mach3_wrapper.get_parameter_names().index('delta_cp')
# weights = cyclical_to_flat_weights(samples[:, dcp_index])


In [9]:
from PIL import Image
import io

def make_posterior_gif(samples_dict, parameter_names, mach3_chain, output_dir="."):
    epochs = sorted(samples_dict.keys())
    dcp_index = parameter_names.index('delta_cp')
    
    os.makedirs(output_dir, exist_ok=True)

    for i, name in tqdm(enumerate(parameter_names), total=len(parameter_names)):
        if "xsec" in name:
            continue
        
        _, bins = np.histogram(mach3_chain[name][10000:], bins=50)
        
        # Pre-pass to find y-axis range
        max_density = 0
        for epoch in epochs:
            s = samples_dict[epoch]
            dcp_samples = s[:, dcp_index].clone()
            dcp_samples_wrapped = (dcp_samples + np.pi) % (2 * np.pi) - np.pi
            weights = cyclical_to_flat_weights(dcp_samples_wrapped)
            
            sample = s[:, i].cpu().numpy()
            if name == 'delta_cp':
                sample = (sample + np.pi) % (2 * np.pi) - np.pi
            
            counts, _ = np.histogram(sample, bins=bins, weights=weights, density=True)
            max_density = max(max_density, counts.max())
        
        mach3_counts, _ = np.histogram(mach3_chain[name][10000:], bins=bins, density=True)
        max_density = max(max_density, mach3_counts.max())

        frames = []
        for epoch in epochs:
            s = samples_dict[epoch]
            
            dcp_samples = s[:, dcp_index].clone()
            dcp_samples_wrapped = (dcp_samples + np.pi) % (2 * np.pi) - np.pi
            weights = cyclical_to_flat_weights(dcp_samples_wrapped)
            
            sample = s[:, i].cpu().numpy()
            if name == 'delta_cp':
                sample = (sample + np.pi) % (2 * np.pi) - np.pi
            
            fig, ax = plt.subplots()
            ax.hist(mach3_chain[name][10000:], bins=bins, density=True,
                    label="MaCh3 Chain", color="k", histtype="step")
            ax.hist(sample, bins=bins, weights=weights, density=True,
                    label=f"ML (epoch {epoch})", histtype="step", color="tab:blue")
            ax.legend(loc='upper right')
            ax.set_xlabel(name)
            ax.set_ylabel("Posterior Density")
            ax.set_title(f"Posterior for {name} — epoch {epoch}")
            ax.set_ylim(0, max_density * 1.1)
            
            buf = io.BytesIO()
            fig.savefig(buf, format='png', dpi=100)
            plt.close(fig)
            buf.seek(0)
            frames.append(Image.open(buf).copy())
        
        gif_path = os.path.join(output_dir, f"posterior_{name}.gif")
        frames[0].save(
            gif_path,
            save_all=True,
            append_images=frames[1:],
            loop=0,
            duration=200
        )
        print(f"Saved: {gif_path}")

parameter_names = inference.simulator.mach3_wrapper.get_parameter_names()
make_posterior_gif(samples, parameter_names, mach3_chain, output_dir="posterior_gifs")

Saved: posterior_gifs/posterior_sin2th_12.gif
Saved: posterior_gifs/posterior_sin2th_23.gif
Saved: posterior_gifs/posterior_sin2th_13.gif
Saved: posterior_gifs/posterior_delm2_12.gif
Saved: posterior_gifs/posterior_delm2_23.gif
Saved: posterior_gifs/posterior_delta_cp.gif


In [10]:
dcp_samples = samples[:, dcp_index].clone()
dcp_samples_wrapped = (dcp_samples + np.pi) % (2 * np.pi) - np.pi  # wrap to (-π, π)

weights = cyclical_to_flat_weights(dcp_samples_wrapped)
print(f"weights min={weights.min():.3f}, max={weights.max():.3f}, mean={weights.mean():.3f}")


NameError: name 'dcp_index' is not defined

In [ ]:
# import numpy as np
# for i, name in tqdm(enumerate(inference.simulator.mach3_wrapper.get_parameter_names()), total=len(inference.simulator.mach3_wrapper.get_parameter_names())):
#     _, bins, _ = plt.hist(mach3_chain[name][10000:], bins=50, density=True, 
#                           label="MaCh3 Chain", color="k", histtype="step")
    
#     sample = samples[:, i].cpu().numpy()
    
#     if name == 'delta_cp':
#         sample = (sample + np.pi) % (2 * np.pi) - np.pi  # wrap for plotting
    
#     plt.hist(sample, bins=bins, weights=weights, density=True, label="ML", histtype="step")
#     plt.legend()
#     plt.xlabel(name)
#     plt.ylabel("Posterior Density")
#     plt.title(f"Comparison of posterior for {name}")
#     plt.show()
#     plt.close()